# MLX Image Classification Benchmarking

An image classification notebook for the `mlx` program

In [1]:
import os

REPO_URL = "https://github.com/ralampay/mlx.git"
BASE_DIR = "/home/ralampay/workspace/mlx/tmp"
REPO_DIR = f"{BASE_DIR}/mlx"

if os.path.exists(REPO_DIR):
  %cd {REPO_DIR}
  !git pull
else:
  !git clone {REPO_URL} {REPO_DIR}
  %cd {REPO_DIR}

!pip install -q -r requirements.txt

/home/ralampay/workspace/mlx/tmp/mlx
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 23 (delta 14), reused 22 (delta 13), pack-reused 0 (from 0)
Unpacking objects: 100% (23/23), 8.88 KiB | 1.78 MiB/s, done.
From https://github.com/ralampay/mlx
   ca850e3..4a2225c  master     -> origin/master
Updating ca850e3..4a2225c
Fast-forward
 README.md                                          |   7 +-
 docs/object_detection/README.md                    | 116 ++++++++++--
 mise.toml                                          |   2 +
 mlx/cli.py                                         |   9 +-
 mlx/modes/object_detection/ultralytics/training.py |  34 ++--
 mlx/modes/object_detection/ultralytics/utils.py    | 152 +++++++++++++--
 notebooks/Intel_Image_Classification.ipynb         | 206 +++++++++++++++++++--
 7 files changed, 463 insertions(+), 63 deletions(-)
 create mode 100644 mise.toml


In [2]:
import os

DATASET_ZIP_URL = "https://happy-research.s3.ap-southeast-1.amazonaws.com/intel-image-classification.zip"
DATA_DIR = f"{BASE_DIR}/datasets"
ZIP_PATH = f"{DATA_DIR}/dataset.zip"
DATASET_NAME = "intel-image-classification"

!mkdir -p {DATA_DIR}

if not os.path.exists(ZIP_PATH):
  !wget -O {ZIP_PATH} "{DATASET_ZIP_URL}"
else:
  print(f"Dataset already downloaded to {ZIP_PATH}")

Dataset already downloaded to /home/ralampay/workspace/mlx/tmp/datasets/dataset.zip


In [3]:
import os
import zipfile

EXTRACT_DIR = f"{DATA_DIR}/image-classification"

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Check if the directory is empty or if it exists with content
# This assumes that if the directory exists and is not empty, extraction has already occurred.
# A more robust check might involve checking for specific files/subdirectories expected after extraction.
if not os.path.exists(EXTRACT_DIR) or not os.listdir(EXTRACT_DIR):
  with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)
  print("Extracted to:", EXTRACT_DIR)
else:
  print(f"Data already extracted to {EXTRACT_DIR}")

print("Top-level contents:", os.listdir(EXTRACT_DIR))

train_dir = EXTRACT_DIR

print("Found class directories in:", train_dir)
print("First 10 class names:", os.listdir(train_dir)[:10])

Data already extracted to /home/ralampay/workspace/mlx/tmp/datasets/image-classification
Top-level contents: ['sea', 'mountain', 'glacier', 'forest', 'buildings', 'street']
Found class directories in: /home/ralampay/workspace/mlx/tmp/datasets/image-classification
First 10 class names: ['sea', 'mountain', 'glacier', 'forest', 'buildings', 'street']


In [4]:
NUM_CLASSES = len(os.listdir(train_dir))
SEED = 27

print(f"Number of classes: {NUM_CLASSES}")

Number of classes: 6


In [5]:
import os
import time

TRAIN_COUNT = 100
VAL_COUNT = 50
TEST_COUNT = 50

timestamp = int(time.time())
OUTPUT_DATASET = f"{BASE_DIR}/{SEED}-{DATASET_NAME}"

if not os.path.exists(OUTPUT_DATASET):
  !python -m mlx \
    --mode image_classification \
    --action build-dataset \
    --dataset "{train_dir}" \
    --output "{OUTPUT_DATASET}" \
    --train-count "{TRAIN_COUNT}" \
    --val-count "{VAL_COUNT}" \
    --test-count "{TEST_COUNT}" \
    --overwrite \
    --seed "{SEED}"

print("Created dataset at:", OUTPUT_DATASET)

Using global random seed=12
╭─────────── MLX ────────────╮
│ Mode: image_classification │
│ Action: build-dataset      │
│ Model: default             │
╰────────────────────────────╯
                     Configuration for resnet18 (standard)                      
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃          Parameter ┃ Value                                                   ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│             action │ build-dataset                                           │
├────────────────────┼─────────────────────────────────────────────────────────┤
│         batch_size │ 1                                                       │
├────────────────────┼─────────────────────────────────────────────────────────┤
│            colored │ True                                                    │
├────────────────────┼─────────────────────────────────────────────────────────┤
│      

In [6]:
EPOCHS = 100
DEVICE = "cuda"
BATCH_SIZE = 16

models = [
    "efficientnet_b0",
    "mobilenet_v3_large",
    "drax_mobilenet_v3_large",
    "densenet121",
    "resnet18",
    "draxnet",
    "resnet50",
    "convnext_tiny",
    "convnext_small",
    "convnext_base"
]

for model_name in models:
    MODEL_NAME = model_name
    RUN_NAME = f"{SEED}-{DATASET_NAME}-{MODEL_NAME}"
    ARTIFACTS_DIR = f"{BASE_DIR}/artifacts/{RUN_NAME}"
    BENCHMARK_DIR = f"{BASE_DIR}/benchmark/{SEED}-{DATASET_NAME}-{MODEL_NAME}"
    CHECKPOINT_PATH = f"{ARTIFACTS_DIR}/{MODEL_NAME}.pth"

    print(f"\n--- Starting training for model: {MODEL_NAME} ---")
    !python -m mlx \
      --mode image_classification \
      --action train \
      --dataset "{OUTPUT_DATASET}" \
      --output "{ARTIFACTS_DIR}" \
      --model "{MODEL_NAME}" \
      --epochs "{EPOCHS}" \
      --batch-size "{BATCH_SIZE}" \
      --device "{DEVICE}" \
      --seed "{SEED}"

    print(f"Training artifacts for {MODEL_NAME} saved to: {ARTIFACTS_DIR}")

    print(f"\n--- Starting benchmarking for model: {MODEL_NAME} ---")
    !mkdir -p "{BENCHMARK_DIR}"

    !python -m mlx \
      --mode image_classification \
      --action benchmark \
      --dataset "{OUTPUT_DATASET}" \
      --model {MODEL_NAME} \
      --model-path "{CHECKPOINT_PATH}" \
      --output "{BENCHMARK_DIR}" \
      --batch-size "{BATCH_SIZE}" \
      --device "{DEVICE}"

    print(f"Benchmark results for {MODEL_NAME} saved to: {BENCHMARK_DIR}")

    import os

    print(f"Benchmark files for {MODEL_NAME}:")
    if os.path.exists(BENCHMARK_DIR):
      for name in sorted(os.listdir(BENCHMARK_DIR)):
        print("-", name)
    else:
      print("  No benchmark directory found.")


--- Starting training for model: efficientnet_b0 ---
Using global random seed=12
╭─────────── MLX ────────────╮
│ Mode: image_classification │
│ Action: train              │
│ Model: efficientnet_b0     │
╰────────────────────────────╯
                  Configuration for efficientnet_b0 (standard)                  
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃          Parameter ┃ Value                                                   ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│             action │ train                                                   │
├────────────────────┼─────────────────────────────────────────────────────────┤
│         batch_size │ 16                                                      │
├────────────────────┼─────────────────────────────────────────────────────────┤
│            colored │ True                                                    │
├────────────────────┼────────────

In [7]:
import os

# Define the source directory to be zipped
source_dir = f"{BASE_DIR}/benchmark"
# Define the output zip file path
output_zip_file = f"{BASE_DIR}/{SEED}-benchmark.zip"

# Ensure the old version is overridden by removing it if it exists
if os.path.exists(output_zip_file):
  !rm -f "{output_zip_file}"
  print(f"Removed existing {output_zip_file}")

# Zip the directory recursively
!zip -r "{output_zip_file}" "{source_dir}"

print(f"Successfully zipped '{source_dir}' to '{output_zip_file}'")

  adding: home/ralampay/workspace/mlx/tmp/benchmark/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/12-intel-image-classification-drax_mobilenet_v3_large/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/12-intel-image-classification-drax_mobilenet_v3_large/confusion_matrix.csv (deflated 35%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/12-intel-image-classification-drax_mobilenet_v3_large/metrics.csv (deflated 28%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/12-intel-image-classification-drax_mobilenet_v3_large/roc_curve.png (deflated 15%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/12-intel-image-classification-drax_mobilenet_v3_large/confusion_matrix.png (deflated 16%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/12-intel-image-classification-convnext_base/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/benchmark/12-intel-image-classification-convnext_base/confusion_matrix.csv (deflated 36%)
  adding: home/ralampay

In [8]:
import os

# Define the source directory to be zipped (the created dataset)
source_dataset_dir = OUTPUT_DATASET
# Define the output zip file path
output_dataset_zip_file = f"{BASE_DIR}/{os.path.basename(OUTPUT_DATASET)}.zip"

# Ensure the old version is overridden by removing it if it exists
if os.path.exists(output_dataset_zip_file):
  !rm -f "{output_dataset_zip_file}"
  print(f"Removed existing {output_dataset_zip_file}")

# Zip the dataset directory recursively
if os.path.exists(source_dataset_dir):
  !zip -r "{output_dataset_zip_file}" "{source_dataset_dir}"
  print(f"Successfully zipped '{source_dataset_dir}' to '{output_dataset_zip_file}'")
else:
  print(f"Dataset directory '{source_dataset_dir}' does not exist. Skipping zip operation.")

  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/sea/ (stored 0%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/sea/2547.jpg (deflated 4%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/sea/7000.jpg (deflated 4%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/sea/18730.jpg (deflated 3%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/sea/3678.jpg (deflated 4%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/sea/2544.jpg (deflated 3%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/sea/16441.jpg (deflated 4%)
  adding: home/ralampay/workspace/mlx/tmp/12-intel-image-classification/train/sea/206.jpg (deflat